# Filter and split the corpus by time period

Removes documents with fewer than 20 tokens, keeps the years covered by the study (1955–2018), and splits the corpus into four 16-year periods. Run this notebook before the per-period preprocessing notebooks.

In [ ]:
import os
import glob
import pandas as pd
import nltk

from nltk.tokenize import word_tokenize

# nltk.download('punkt')

### Remove documents with fewer than 20 tokens

In [ ]:
ip_df_english =pd.read_pickle("../../datasets/processed/ip_df_english_processed_full.pkl")
ip_df_english["num_tokens"] = ip_df_english["content_translated_processed"].apply(lambda content: len(word_tokenize(content)))

ip_df_english_filtered = ip_df_english[ip_df_english["num_tokens"] >= 20].copy()

### Keep the study period (1955–2018)

In [ ]:
ip_df_english_filtered = ip_df_english_filtered[ip_df_english_filtered["year"].astype(int) >= 1955].copy()
ip_df_english_filtered.to_pickle("../../datasets/processed/ip_df_english_filtered.pkl")

In [ ]:
# Documents containing atleast 20 tokens
ip_df_english_filtered = pd.read_pickle("../../datasets/processed/ip_df_english_filtered.pkl")

In [ ]:
# calculate documents per year
documents_per_year =ip_df_english_filtered.groupby('year').size().reset_index(name='number_of_documents')

### Split the corpus into time periods

In [ ]:
output_dir ="../../datasets/data_split_timeframe/"
os.makedirs(output_dir, exist_ok=True)

# Create a list of unique years and sort them
unique_years = sorted(ip_df_english_filtered['year'].unique())

# Split the unique years into groups of 16 years each
year_groups = [unique_years[i:i+16] for i in range(0, len(unique_years), 16)]

# Iterate over each group of 16 years
for i, years in enumerate(year_groups):
    # Get the start and end years for the current group
    start_year = years[0]
    end_year = years[-1]

    # Create a subset dataframe containing documents within the current year range
    subset_df = ip_df_english_filtered[ip_df_english_filtered['year'].isin(years)]

    # Print the number of documents in the current subset
    print(f"Subset {i+1} ({start_year}-{end_year}): {len(subset_df)} documents")

    # Save the smaller subset as a .pkl file with the year range as the filename
    filename = os.path.join(output_dir, f"data_{start_year}-{end_year}.pkl")
    subset_df.to_pickle(filename)

In [ ]:
# Check the number of documents in each subset
folder_path = "../../datasets/data_split_timeframe/"
total_documents = 0

for file_path in sorted(glob.glob(os.path.join(folder_path, "data_*-*.pkl"))):
    df = pd.read_pickle(file_path)
    print(f"File: {os.path.basename(file_path)}, Number of documents: {len(df)}")
    total_documents += len(df)

print("Total documents:", total_documents)